# 01 — Columns, types and nulls

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)

spark = (
    SparkSession.builder
    .appName("Exercise-01")
    .master(master_url)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
raw = spark.createDataFrame([
    (1,' Mug ','19.99','2','Kitchen'), (2,'Bag','bad','3',None),
    (3,'Notebook',None,'1','Stationery'), (4,'Lamp','14.50',None,'Lighting')],
    'id INT, name STRING, price_text STRING, qty_text STRING, category STRING')
raw.show(truncate=False)
raw.printSchema()

## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Cast dirty numeric fields

Trim name; cast price/quantity without crashing on bad strings. Keep id/name/price/qty/category.

**Expected result:** Four rows; bad and missing prices become null, not zero.

In [ ]:
typed = None  # TODO: use try_cast, aliases, and trim

**Hint 1:** Invalid strings and nulls are different input defects but can both yield null after conversion.

**Hint 2:** Use F.expr("try_cast(price_text AS DOUBLE)") and .alias("price"); use trim for names.

**Hint 3:** Start raw.select("id", F.trim("name").alias("name"), ...). Avoid inferSchema for this dirty input.

### Check

In [ ]:
check_frame(typed,['id','name','price','qty'],[(1,'Mug',19.99,2),(2,'Bag',None,3),(3,'Notebook',None,1),(4,'Lamp',14.5,None)])

## Task 2: Apply sensible defaults

Fill missing category with Unknown and missing quantity with 0. Do not replace missing prices with 0.

**Expected result:** Only category and quantity are filled.

In [ ]:
filled = None  # TODO: use typed.na.fill with a dictionary

**Hint 1:** A fill policy is a business decision; unknown price should stay unknown.

**Hint 2:** na.fill accepts a mapping by output column name.

**Hint 3:** Use typed.na.fill({"category": ..., "qty": ...}); retain every row.

### Check

In [ ]:
check_frame(filled,['id','category','qty','price'],[(1,'Kitchen',2,19.99),(2,'Unknown',3,None),(3,'Stationery',1,None),(4,'Lighting',0,14.5)])

## Task 3: Flag valid sales and calculate value

Add valid_sale=(price not null and qty>0). line_total is price*qty only for valid sales; otherwise null.

**Expected result:** Only Mug has a valid total: 39.98.

In [ ]:
priced = None  # TODO: withColumn, when/otherwise

**Hint 1:** Multiplication propagates unknowns; a default quantity of zero does not mean a valid sale.

**Hint 2:** Combine Column expressions with & and parentheses, not Python and.

**Hint 3:** filled.withColumn("valid_sale", ...).withColumn("line_total", F.when(..., ...)).

### Check

In [ ]:
check_frame(priced,['id','valid_sale','line_total'],[(1,True,39.98),(2,False,None),(3,False,None),(4,False,None)])

## More practice

Rename price to unit_price; drop unused input columns; compare count before and after.

## Common mistakes

Do not use Python and on Columns. try_cast differs from a strict ANSI cast. Schema inference is not validation.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()